# QLoRA fine-tune: Qwen3-4B -> APUSH item writer (v5 semantic preservation)

Distills the independently audited expert-anchor dataset (`data/generated/train_sft_v5.jsonl`) into Qwen3-4B, training **only on the JSON assistant response** while masking the prompt.

**Runtime:** Colab GPU. A free **T4** is enough for this 4B QLoRA run. Set `Runtime -> Change runtime type -> GPU`.

**Data:** clones the repo to read `train_sft_v5.jsonl`. If the repo is private, use the upload fallback in the dataset cell.

**Run naming:** local adapters, GGUF exports, and Hugging Face artifacts derive from `RUN_NAME = "qwen3-4b-apush-v5-semantic-preservation"`, while checkpoint folders also include the dataset hash so incompatible runs cannot resume each other.

Pipeline: install -> load 4-bit model -> add LoRA -> load+template data -> train (loss on response) -> sanity-generate -> save adapter (+ optional GGUF for Ollama).


In [ ]:
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "CPU"
print(DEVICE_NAME, "| CUDA available:", CUDA_AVAILABLE)

if not CUDA_AVAILABLE:
    raise RuntimeError("Enable a GPU runtime before loading the 4-bit model.")


In [ ]:
# 1) Install Unsloth (pulls transformers/trl/peft/bitsandbytes). torchao is required
#    by unsloth_zoo, but only run the fallback lines below if the import/load cell fails.
%pip install -q unsloth
%pip install -q --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Diagnostics: if the load cell crashes on torchao, capture these versions.
import importlib.metadata as md
import torch

def _version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return "not installed"

print(
    "torch:", torch.__version__,
    "| torchao:", _version("torchao"),
    "| transformers:", _version("transformers"),
)

# FALLBACK ONLY if the next cell errors on torchao `_wrap_tensor_autograd`.
# Uncomment one line, then Runtime -> Restart session -> Run All.
# %pip install -q --upgrade torchao
# %pip install -q --upgrade "torch>=2.6" torchao
# %pip install -q "torchao==0.9.0"


In [ ]:
# 2) Load Qwen3-4B in 4-bit. Keep v5 run/artifact names centralized here.
from unsloth import FastLanguageModel
import torch

BASE_MODEL_NAME = "unsloth/Qwen3-4B"
BASE_MODEL_REVISION = "64033659d5caf1b8ed7f929b29de705e93a4d468"  # immutable HF commit; keep eval aligned
BASE_MODEL_USE_EXACT_NAME = True  # keep this revision attached to its original repo; disable Unsloth mirror remapping
RUN_NAME = "qwen3-4b-apush-v5-semantic-preservation"
MAX_SEQ_LEN = 4096                      # prompt (~2.1k tok) + JSON completion fits comfortably
SEED = 42
EXPECTED_DATASET_SHA256 = "06cfe7196256efaa94aca36550355b67deb54b42c418f9037e24cce7f0a21e44"

HF_REPO_ID = f"rohanpalviela/{RUN_NAME}-lora"
ADAPTER_DIR = f"{RUN_NAME}-lora"
GGUF_SAVE_DIR = f"{RUN_NAME}-gguf"
REPO_URL = "https://github.com/RohanPalivela/slm.git"
REPO_DIR = "slm"
REPO_REF = "main"

BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    revision=BASE_MODEL_REVISION,
    use_exact_model_name=BASE_MODEL_USE_EXACT_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,      # auto (bf16 on Ampere+, fp16 on T4)
)


In [ ]:
# 3) Attach a lower-capacity LoRA adapter to reduce semantic drift on the compact v5 set.
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)


In [ ]:
# 4) Get the dataset. Refresh an existing Colab clone so stale data cannot be trained.
import hashlib
import os
import subprocess
from pathlib import Path

if os.path.exists(REPO_DIR) and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    raise RuntimeError(f"{REPO_DIR} exists but is not a git clone; rename it before continuing")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_REF])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"])

DATA = os.path.join(REPO_DIR, "data/generated/train_sft_v5.jsonl")

# Option B (private repo): comment the clone above and upload the file instead.
# from google.colab import files
# up = files.upload()
# DATA = next(iter(up))

if not os.path.exists(DATA):
    raise FileNotFoundError(f"Dataset not found: {DATA}")
DATASET_SHA256 = hashlib.sha256(Path(DATA).read_bytes()).hexdigest()
if DATASET_SHA256 != EXPECTED_DATASET_SHA256:
    raise RuntimeError(f"V5 dataset hash mismatch: expected {EXPECTED_DATASET_SHA256}, found {DATASET_SHA256}. Push or select the exact v5 data revision before training.")
RUN_INSTANCE_NAME = f"{RUN_NAME}-{DATASET_SHA256[:8]}"
print("dataset:", DATA)
print("dataset sha256:", DATASET_SHA256)
print("run instance:", RUN_INSTANCE_NAME)


In [ ]:
# 5) Apply the Qwen3 chat template to each example -> a single `text` field.
#    Qwen3 may insert an empty <think></think> wrapper even with enable_thinking=False;
#    strip that wrapper so the trained assistant span is JSON only.
from datasets import load_dataset
import re

raw_dataset = load_dataset("json", data_files=DATA, split="train")

# Hold out complete sources, not random rows. Random row splitting leaks the
# same stimulus into train and validation because each source has many items.
import random
from collections import defaultdict
rng = random.Random(SEED)
source_genres = {}
for source_id, genre in zip(raw_dataset["source_id"], raw_dataset["source_genre"]):
    source_genres[source_id] = genre
sources_by_genre = defaultdict(list)
for source_id, genre in source_genres.items():
    sources_by_genre[genre].append(source_id)
val_source_ids = set()
for genre, source_ids in sorted(sources_by_genre.items()):
    rng.shuffle(source_ids)
    n_genre_val = max(1, round(0.12 * len(source_ids)))
    val_source_ids.update(source_ids[:n_genre_val])
train_raw = raw_dataset.filter(lambda row: row["source_id"] not in val_source_ids)
validation_raw = raw_dataset.filter(lambda row: row["source_id"] in val_source_ids)


def strip_empty_think_block(text):
    return re.sub(r"<think>\s*</think>\s*", "", text)


def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": strip_empty_think_block(text)}


train_dataset = train_raw.map(to_text, remove_columns=train_raw.column_names)
validation_dataset = validation_raw.map(to_text, remove_columns=validation_raw.column_names)
print(train_dataset)
print("train examples:", len(train_dataset), "| validation examples:", len(validation_dataset))
print("held-out validation sources:", sorted(val_source_ids))
print(train_dataset[0]["text"][:600])


In [ ]:
# 5b) RESILIENCE: checkpoint to Google Drive so a Colab disconnect can resume.
#     Colab free T4 sessions drop periodically; local /content is wiped on reset.
import os

USE_DRIVE = True   # set False to keep checkpoints only on the ephemeral Colab disk

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = f"/content/drive/MyDrive/{RUN_INSTANCE_NAME}-outputs"
else:
    OUTPUT_DIR = f"{RUN_INSTANCE_NAME}-outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("checkpoints ->", OUTPUT_DIR)


In [ ]:
# 6) Trainer. train_on_responses_only masks the prompt so loss is on the JSON only.
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,   # effective batch 4; one low-intensity pass over v5
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=4e-5,  # sharply reduce cumulative update pressure after the v4 semantic regression
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=SEED,
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
        save_total_limit=1,
        eval_strategy="epoch",
        load_best_model_at_end=False,
        report_to="none",
        bf16=BF16,
        fp16=not BF16,
    ),
)

# Qwen3 uses ChatML turn markers.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


In [ ]:
# 7) Train. Auto-resume from the newest checkpoint if a prior run was cut off.
#    In a fresh runtime, re-run cells 1-6 first; this then picks up where it stopped.
import os
import re


def latest_checkpoint(directory):
    if not os.path.isdir(directory):
        return None
    checkpoints = []
    for name in os.listdir(directory):
        match = re.fullmatch(r"checkpoint-(\d+)", name)
        path = os.path.join(directory, name)
        if match and os.path.isdir(path):
            checkpoints.append((int(match.group(1)), path))
    return max(checkpoints)[1] if checkpoints else None


def strip_empty_think_block(text):
    return re.sub(r"<think>\s*</think>\s*", "", text).strip()


# Response-mask sanity check: the non-masked labels should decode to assistant JSON only.
mask_sample = trainer.train_dataset[0]
label_ids = mask_sample["labels"]
input_ids = mask_sample["input_ids"]
trained_ids = [int(tok) for tok, label in zip(input_ids, label_ids) if int(label) != -100]
trained_text = tokenizer.decode(trained_ids, skip_special_tokens=True).strip()
json_text = strip_empty_think_block(trained_text)
print("raw trained span preview:")
print(trained_text[:1000])
if json_text != trained_text:
    print("stripped empty Qwen3 think wrapper; JSON span preview:")
    print(json_text[:1000])
assert json_text.startswith("[") and '"archetype"' in json_text, "response mask is not isolating assistant JSON"

ckpt = latest_checkpoint(OUTPUT_DIR)
print(f"resuming from {ckpt}" if ckpt else "fresh run (no checkpoint found)")
stats = trainer.train(resume_from_checkpoint=ckpt)
print(stats)

# Persist enough provenance to reproduce the exact adapter and compare checkpoints.
import hashlib
import json
import subprocess
from pathlib import Path
run_metadata = {
    'run_name': RUN_NAME, 'run_instance_name': RUN_INSTANCE_NAME, 'base_model': BASE_MODEL_NAME, 'base_revision': BASE_MODEL_REVISION,
    'base_model_use_exact_name': BASE_MODEL_USE_EXACT_NAME,
    'seed': SEED, 'max_seq_len': MAX_SEQ_LEN, 'train_examples': len(train_dataset),
    'validation_examples': len(validation_dataset), 'validation_source_ids': sorted(val_source_ids), 'train_source_ids': sorted(set(train_raw['source_id'])),
    'dataset_path': DATA, 'dataset_sha256': DATASET_SHA256, 'dataset_version': 'v5',
    'selection_policy': 'independent_current_rubric_expert_curated_only_v1',
    'training_objective': 'semantic_preservation_with_contract_learning',
    'prompt_sha256': hashlib.sha256((Path(REPO_DIR) / 'prompts/litmus_generation_prompt.md').read_bytes()).hexdigest(),
    'archetype_policy_sha256': hashlib.sha256((Path(REPO_DIR) / 'data/training_archetype_policy.json').read_bytes()).hexdigest(),
    'repo_revision': subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip(),
    'training': {'epochs': 1, 'learning_rate': 4e-5, 'effective_batch_size': 4, 'lora_r': 8, 'lora_alpha': 16, 'checkpoint_strategy': 'epoch', 'validation_split': 'source_stratified_by_genre'},
    'trainer_metrics': dict(stats.metrics),
    'packages': {name: _version(name) for name in ['unsloth','transformers','trl','peft','torch','bitsandbytes']},
}
RUN_METADATA_PATH = Path(OUTPUT_DIR) / 'training_run_metadata.json'
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
print('run metadata ->', RUN_METADATA_PATH)


In [ ]:
# 8) Sanity-generate on a held-out source (not in TRAIN) to eyeball tuned output.
import os
import sys
import torch

sys.path.insert(0, REPO_DIR)
from eval.prompt_loader import LitmusPrompt, generation_format_diagnostics
from transformers import StoppingCriteria, StoppingCriteriaList

class StopAfterCompleteJsonArray(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length
    def __call__(self, input_ids, scores, **kwargs):
        generated = "[" + self.tokenizer.decode(input_ids[0, self.prompt_length:], skip_special_tokens=True)
        return generation_format_diagnostics(generated)["strict_array_contract"]

FastLanguageModel.for_inference(model)

prompt = LitmusPrompt.from_file(os.path.join(REPO_DIR, "prompts/litmus_generation_prompt.md"))
source = "Federalist No. 10 ... the tendency to break and control the violence of faction ..."
attribution = "James Madison, Federalist No. 10, 1787"
note = "Federalist arguments for an extended republic as a response to faction."
system_prompt, user_prompt = prompt.build(
    source=source,
    attribution=attribution,
    note=note,
    n=1,
    archetypes="CAUSE_OF_SOURCE",
    difficulty="operational / test-day",
    include_fewshot=False,
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]
rendered = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
rendered = strip_empty_think_block(rendered).rstrip() + "["
assert "<think" not in rendered.lower(), "sanity prompt still contains a thinking prefill"
ids = tokenizer(rendered, return_tensors="pt")["input_ids"].to("cuda")

with torch.inference_mode():
    out = model.generate(
        input_ids=ids,
        max_new_tokens=768,
        temperature=0.7,
        do_sample=True,
        stopping_criteria=StoppingCriteriaList([StopAfterCompleteJsonArray(tokenizer, ids.shape[1])]),
        pad_token_id=tokenizer.eos_token_id,
    )

print("[" + tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))


In [ ]:
# 9) Save adapter + export GGUF, then persist artifacts so Colab cannot eat them.
#    The GGUF is what Ollama needs for base-vs-tuned eval.
import glob
import os
import shutil
from huggingface_hub import HfApi, login

SAVE_GGUF_TO_DRIVE = USE_DRIVE
UPLOAD_ADAPTER_TO_HF = True
UPLOAD_GGUF_TO_HF = True

# Save the LoRA adapter locally.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved adapter ->", ADAPTER_DIR)

api = HfApi()
if UPLOAD_ADAPTER_TO_HF or UPLOAD_GGUF_TO_HF:
    login()
    api.create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)

if UPLOAD_ADAPTER_TO_HF:
    model.push_to_hub(HF_REPO_ID, token=True)
    tokenizer.push_to_hub(HF_REPO_ID, token=True)
    if RUN_METADATA_PATH.exists():
        api.upload_file(path_or_fileobj=str(RUN_METADATA_PATH), path_in_repo='training_run_metadata.json', repo_id=HF_REPO_ID)
    print("uploaded adapter ->", HF_REPO_ID)

# Export the merged model to q4_k_m GGUF for Ollama.
export_info = model.save_pretrained_gguf(
    GGUF_SAVE_DIR,
    tokenizer,
    quantization_method="q4_k_m",
) or {}
print(export_info)

gguf_dir = export_info.get("gguf_directory") or f"{GGUF_SAVE_DIR}_gguf"
gguf_files = export_info.get("gguf_files") or sorted(glob.glob(os.path.join(gguf_dir, "*.gguf")))
modelfile = export_info.get("modelfile_location") or os.path.join(gguf_dir, "Modelfile")
if not gguf_files:
    raise FileNotFoundError(f"No GGUF files found in {gguf_dir}; export did not finish cleanly.")
print("GGUF files:", gguf_files)
print("Modelfile:", modelfile)

# Persistent copy in Google Drive. This survives Colab runtime resets.
if SAVE_GGUF_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        drive_dir = f"/content/drive/MyDrive/{RUN_NAME}-gguf"
        shutil.copytree(gguf_dir, drive_dir, dirs_exist_ok=True)
        print("copied GGUF folder to Drive:", drive_dir)
    except Exception as exc:
        print("WARNING: Drive copy failed:", repr(exc))

# Upload the Ollama-ready artifacts to Hugging Face.
if UPLOAD_GGUF_TO_HF:
    for path in [*gguf_files, modelfile]:
        if not path or not os.path.exists(path):
            print("skipping missing artifact:", path)
            continue
        path_in_repo = os.path.basename(path)
        print(f"uploading {path} -> {HF_REPO_ID}/{path_in_repo}")
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=path_in_repo,
            repo_id=HF_REPO_ID,
        )
    print("uploaded GGUF artifacts ->", HF_REPO_ID)

print("Next on your Mac:")
print(f"  python3 scripts/register_ollama_from_hf.py --repo {HF_REPO_ID} --local-dir models/qwen3-apush-v5-semantic-preservation --model-name qwen3-apush-v5-semantic-preservation:latest")


## Next: matched base-vs-v5 evaluation

1. Upload the adapter and its `training_run_metadata.json` from cell 9.
2. Run `notebooks/eval_hf_gpu.ipynb` from the top at the exact pushed repository commit.
3. Keep the frozen 14-source cohort and two matched candidate repetitions unchanged; each model receives at most four total generation attempts per prompt.
4. Preserve the downloaded ZIP before deciding whether to export or register a GGUF.
